# scATAC-seq Allele-Specific Chromatin Accessibility Analysis using EM Algorithm

This notebook implements an Expectation-Maximization (EM) algorithm to analyze single-cell ATAC-seq data from hybrid mouse embryonic stem cells (F1 CAST×B6). The algorithm identifies three cell states based on allele-specific chromatin accessibility patterns at replication origins:

1. **CAST > B6**: Cells with preferential chromatin accessibility on CAST alleles
2. **B6 > CAST**: Cells with preferential chromatin accessibility on B6 alleles  
3. **Noise**: Low-quality cells or cells without clear allelic preference

## Citation

If you use this code, please cite:
[Your paper citation here]

## Requirements

- Python 3.7+
- NumPy
- Pandas
- SciPy
- Google Colab (for Drive mounting; optional if running locally)

## Table of Contents

1. [Setup and Installation](#setup)
2. [Core EM Algorithm](#em-algorithm)
3. [Data Loading and Analysis](#data-analysis)
4. [Control Analysis (Annotation Shuffling)](#control-analysis)
5. [Results and Output](#results)

---
## 1. Setup and Installation {#setup}

Import required libraries and mount Google Drive (if using Colab).

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
from scipy.stats import poisson
from scipy.special import logsumexp

# Mount Google Drive (comment out if running locally)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## 2. Core EM Algorithm {#em-algorithm}

### Algorithm Overview

The EM algorithm models each genomic region's read counts using a Poisson distribution with region- and state-specific rate parameters (λ). The algorithm:

1. **E-step**: Calculates the probability (responsibility) that each cell belongs to each state
2. **M-step**: Updates the state parameters based on the responsibilities
3. Iterates until convergence

### Key Features

- **Three-state model**: CAST>B6, B6>CAST, and Noise states
- **Noise detection**: Pre-identifies low-quality cells based on signal characteristics
- **State separation**: Enforces minimum separation between biological states to prevent collapse
- **Controlled mixing**: Maintains target proportions while allowing data-driven adjustment

In [ ]:
def em_algorithm(data, region_types, n_states=3, max_iter=200,
                 convergence_threshold=1e-8, target_noise_prop=0.15):
    """
    Expectation-Maximization algorithm for allele-specific chromatin accessibility analysis.

    Parameters:
    -----------
    data : numpy.ndarray
        Shape (n_cells, n_regions). Read counts for each cell at each genomic region.
    region_types : numpy.ndarray
        Shape (n_regions,). Binary labels: 1 for CAST-early regions, 0 for B6-early regions.
    n_states : int
        Number of states (default: 3 for CAST>B6, B6>CAST, Noise)
    max_iter : int
        Maximum number of EM iterations (default: 200)
    convergence_threshold : float
        Relative change in log-likelihood for convergence (default: 1e-8)
    target_noise_prop : float
        Target proportion for noise state (default: 0.15)

    Returns:
    --------
    mixing_proportions : numpy.ndarray
        Shape (n_states,). Estimated proportion of cells in each state.
    lambda_params : numpy.ndarray
        Shape (n_states, n_regions). Poisson rate parameters for each state and region.
    responsibilities : numpy.ndarray
        Shape (n_cells, n_states). Probability of each cell belonging to each state.
    """
    n_cells, n_regions = data.shape

    # Initialize mixing proportions
    target_props = np.array([0.45, 0.45, 0.1])
    mixing_proportions = target_props.copy()

    # Calculate region statistics for initialization
    region_medians = np.median(data, axis=0)
    region_q75 = np.percentile(data, 75, axis=0)
    region_q25 = np.percentile(data, 25, axis=0)

    # Initialize lambda parameters with strong separation
    lambda_params = np.zeros((n_states, n_regions))
    cast_early_mask = region_types == 1
    b6_early_mask = ~cast_early_mask

    # State 0: CAST > B6
    lambda_params[0, cast_early_mask] = region_q75[cast_early_mask] * 1.2
    lambda_params[0, b6_early_mask] = region_q25[b6_early_mask] * 0.8

    # State 1: B6 > CAST
    lambda_params[1, cast_early_mask] = region_q25[cast_early_mask] * 0.8
    lambda_params[1, b6_early_mask] = region_q75[b6_early_mask] * 1.2

    # State 2: Noise (uniform low signal)
    lambda_params[2] = region_medians * 0.5

    # Pre-identify noise candidate cells
    total_signal = np.median(data, axis=1)
    q75_signal = np.percentile(data, 75, axis=1)
    q25_signal = np.percentile(data, 25, axis=1)
    signal_spread = (q75_signal - q25_signal) / (total_signal + 1e-6)
    noise_score = signal_spread * (1 / (total_signal + 1e-6))
    noise_threshold = np.percentile(noise_score, 85)
    noise_candidates = noise_score > noise_threshold

    log_likelihood_old = -np.inf

    # EM iterations
    for iteration in range(max_iter):
        # ================== E-STEP ==================
        log_responsibilities = np.zeros((n_cells, n_states))

        for k in range(n_states):
            # Calculate log-likelihood under Poisson model
            log_poisson = np.sum(poisson.logpmf(data, lambda_params[k]), axis=1)
            log_responsibilities[:, k] = np.log(mixing_proportions[k]) + log_poisson

        # Increase preference for noise state in identified noise candidates
        log_responsibilities[noise_candidates, 2] += 1.5

        # Balance biological states to prevent extreme assignments
        for i in range(n_cells):
            if not noise_candidates[i]:
                max_resp = max(log_responsibilities[i, 0], log_responsibilities[i, 1])
                min_resp = min(log_responsibilities[i, 0], log_responsibilities[i, 1])
                if np.isfinite(max_resp) and np.isfinite(min_resp):
                    if max_resp - min_resp > 2.5:
                        if log_responsibilities[i, 0] > log_responsibilities[i, 1]:
                            log_responsibilities[i, 1] = max_resp - 2.0
                        else:
                            log_responsibilities[i, 0] = max_resp - 2.0

        # Normalize responsibilities using log-sum-exp for numerical stability
        log_norm = logsumexp(log_responsibilities, axis=1, keepdims=True)
        log_responsibilities -= log_norm
        responsibilities = np.exp(log_responsibilities)

        # Handle numerical issues
        if np.any(~np.isfinite(responsibilities)):
            responsibilities = np.nan_to_num(responsibilities, nan=1.0/n_states)

        # ================== M-STEP ==================
        # Update mixing proportions with controlled adjustment
        raw_proportions = responsibilities.mean(axis=0)
        mixing_proportions = 0.7 * target_props + 0.3 * raw_proportions
        mixing_proportions /= mixing_proportions.sum()

        # Update lambda parameters
        for k in range(n_states):
            weighted_sum = np.sum(responsibilities[:, k][:, np.newaxis] * data, axis=0)
            total_weight = responsibilities[:, k].sum()

            if k < 2:  # Biological states
                new_lambdas = weighted_sum / max(total_weight, 1e-10)

                if k == 0:  # CAST > B6
                    lambda_params[k, cast_early_mask] = (0.8 * new_lambdas[cast_early_mask] +
                                                          0.2 * region_q75[cast_early_mask] * 1.2)
                    lambda_params[k, b6_early_mask] = (0.8 * new_lambdas[b6_early_mask] +
                                                        0.2 * region_q25[b6_early_mask] * 0.8)
                else:  # B6 > CAST
                    lambda_params[k, cast_early_mask] = (0.8 * new_lambdas[cast_early_mask] +
                                                          0.2 * region_q25[cast_early_mask] * 0.8)
                    lambda_params[k, b6_early_mask] = (0.8 * new_lambdas[b6_early_mask] +
                                                        0.2 * region_q75[b6_early_mask] * 1.2)
            else:  # Noise state - uniform across regions
                lambda_params[k] = np.median(weighted_sum / max(total_weight, 1e-10)) * np.ones_like(lambda_params[k])

        # Enforce minimum separation between biological states
        min_diff = 0.4

        # State 0: CAST > B6
        cast_med_0 = np.median(lambda_params[0, cast_early_mask])
        b6_med_0 = np.median(lambda_params[0, b6_early_mask])
        if (cast_med_0 - b6_med_0) < min_diff:
            adjustment = (min_diff - (cast_med_0 - b6_med_0)) * 0.6
            lambda_params[0, cast_early_mask] += adjustment
            lambda_params[0, b6_early_mask] -= adjustment

        # State 1: B6 > CAST
        cast_med_1 = np.median(lambda_params[1, cast_early_mask])
        b6_med_1 = np.median(lambda_params[1, b6_early_mask])
        if (b6_med_1 - cast_med_1) < min_diff:
            adjustment = (min_diff - (b6_med_1 - cast_med_1)) * 0.6
            lambda_params[1, b6_early_mask] += adjustment
            lambda_params[1, cast_early_mask] -= adjustment

        # Ensure all lambdas are positive
        lambda_params = np.maximum(lambda_params, 1e-6)

        # ================== CONVERGENCE CHECK ==================
        log_likelihood_new = np.sum(log_norm)
        if iteration > 0 and log_likelihood_old > -np.inf:
            rel_improvement = abs((log_likelihood_new - log_likelihood_old) /
                                abs(log_likelihood_old))
            if rel_improvement < convergence_threshold:
                print(f"Converged after {iteration + 1} iterations")
                break

        log_likelihood_old = log_likelihood_new

    return mixing_proportions, lambda_params, responsibilities


def analyze_atac_seq_data(data, region_types):
    """
    Analyze scATAC-seq data and print detailed statistics about identified states.

    Parameters:
    -----------
    data : numpy.ndarray
        Shape (n_cells, n_regions). Read counts for each cell at each genomic region.
    region_types : numpy.ndarray
        Shape (n_regions,). Binary labels: 1 for CAST-early, 0 for B6-early.

    Returns:
    --------
    mixing_proportions : numpy.ndarray
        Estimated proportion of cells in each state.
    lambda_params : numpy.ndarray
        Poisson rate parameters for each state and region.
    responsibilities : numpy.ndarray
        Probability of each cell belonging to each state.
    cell_states : numpy.ndarray
        Hard assignment of each cell to its most likely state (0, 1, or 2).
    """
    # Run EM algorithm
    mixing_proportions, lambda_params, responsibilities = em_algorithm(data, region_types)

    # Assign cells to states based on maximum responsibility
    cell_states = np.argmax(responsibilities, axis=1)
    max_probs = responsibilities.max(axis=1)

    # Separate CAST and B6 regions
    cast_early_regions = region_types == 1
    b6_early_regions = ~cast_early_regions

    print("\nRobust State Analysis:")
    print("=" * 60)

    # Print statistics for each state
    for i in range(3):
        state_name = "CAST > B6" if i == 0 else "B6 > CAST" if i == 1 else "Noise"
        print(f"\nState {i+1} ({state_name}):")

        # CAST region statistics
        cast_vals = lambda_params[i, cast_early_regions]
        percentiles = np.percentile(cast_vals, [10, 25, 50, 75, 90])
        print(f"CAST regions:")
        print(f"  Median: {percentiles[2]:.3f}")
        print(f"  IQR: {percentiles[3] - percentiles[1]:.3f}")
        print(f"  10-90 range: [{percentiles[0]:.3f}, {percentiles[4]:.3f}]")
        print(f"  Q1-Q3: [{percentiles[1]:.3f}, {percentiles[3]:.3f}]")

        # B6 region statistics
        b6_vals = lambda_params[i, b6_early_regions]
        percentiles = np.percentile(b6_vals, [10, 25, 50, 75, 90])
        print(f"B6 regions:")
        print(f"  Median: {percentiles[2]:.3f}")
        print(f"  IQR: {percentiles[3] - percentiles[1]:.3f}")
        print(f"  10-90 range: [{percentiles[0]:.3f}, {percentiles[4]:.3f}]")
        print(f"  Q1-Q3: [{percentiles[1]:.3f}, {percentiles[3]:.3f}]")

        # Calculate median difference for biological states
        if i < 2:
            diff = (np.median(cast_vals) - np.median(b6_vals)) if i == 0 else (np.median(b6_vals) - np.median(cast_vals))
            print(f"Median difference: {diff:.3f}")

    # Print cell distribution across states
    print(f"\nCell Distribution:")
    print("=" * 60)
    for i in range(3):
        state_cells = cell_states == i
        n_cells = state_cells.sum()
        prop = state_cells.mean()
        med_prob = np.median(max_probs[state_cells]) if np.any(state_cells) else 0
        iqr_prob = (np.percentile(max_probs[state_cells], 75) -
                    np.percentile(max_probs[state_cells], 25)) if np.any(state_cells) else 0
        state_name = "CAST > B6" if i == 0 else "B6 > CAST" if i == 1 else "Noise"
        print(f"State {i+1} ({state_name}): {n_cells} cells ({prop:.1%})")
        print(f"  Median confidence: {med_prob:.3f}")
        print(f"  IQR confidence: {iqr_prob:.3f}")

    return mixing_proportions, lambda_params, responsibilities, cell_states

---
## 3. Data Loading and Analysis {#data-analysis}

### Input Data Format

The analysis requires two input files:

1. **Accessibility matrix** (`*.csv`):
   - Rows: individual cells
   - Columns: genomic regions (replication origins)
   - Values: read counts (scATAC-seq signal)
   - First column: cell identifiers (row names)

2. **Region types** (`*.csv`):
   - Single column with binary labels for each region
   - 1 = CAST-early replication origin
   - 0 = B6-early replication origin
   - No header, same order as columns in accessibility matrix

### File Paths

**Important**: Update the file paths below to match your data location.

In [ ]:
# ============= CONFIGURE YOUR FILE PATHS HERE =============

# Path to accessibility matrix (cells × regions)
DATA_PATH = '/content/drive/MyDrive/c_merged_bi_ESC_new.csv'

# Path to region type labels (1=CAST-early, 0=B6-early)
REGION_TYPES_PATH = '/content/drive/MyDrive/regions_numeric_merged_bi_ESC_new.csv'

# Output directory for results
OUTPUT_DIR = '/content/drive/MyDrive/'

# ==========================================================

# Load accessibility data
print("Loading data...")
data_df = pd.read_csv(DATA_PATH, index_col=0)
data = data_df.values
cell_names = data_df.index.tolist()

print(f"Data shape: {data.shape[0]} cells × {data.shape[1]} regions")
print(f"Cell names (first 5): {cell_names[:5]}")

# Load region types
region_types = np.loadtxt(REGION_TYPES_PATH, delimiter=',', dtype=int)
print(f"Region types shape: {region_types.shape}")
print(f"CAST-early regions: {(region_types == 1).sum()}")
print(f"B6-early regions: {(region_types == 0).sum()}")

Loading data...
Data shape: 9658 cells × 610 regions
Cell names (first 5): ['AAACGAAAGAACTCCT-1', 'AAACGAAAGAGCGGTT-1', 'AAACGAAAGATGTTCC-1', 'AAACGAAAGGGCGAAG-1', 'AAACGAAAGGTCACTT-1']
Region types shape: (610,)
CAST-early regions: 305
B6-early regions: 305


### Run EM Algorithm

Execute the EM algorithm on the real data with correct region annotations.

In [ ]:
# Run analysis
print("Running EM algorithm...")
mixing_proportions, lambda_params, responsibilities, cell_states = analyze_atac_seq_data(
    data=data,
    region_types=region_types
)

Running EM algorithm...


/tmp/ipython-input-2721987829.py:95: RuntimeWarning: invalid value encountered in subtract
  log_responsibilities -= log_norm



Robust State Analysis:

State 1 (CAST > B6):
CAST regions:
  Median: 0.556
  IQR: 1.171
  10-90 range: [0.273, 3.007]
  Q1-Q3: [0.344, 1.515]
B6 regions:
  Median: 0.075
  IQR: 0.644
  10-90 range: [0.000, 1.506]
  Q1-Q3: [0.000, 0.644]
Median difference: 0.481

State 2 (B6 > CAST):
CAST regions:
  Median: 0.090
  IQR: 0.591
  10-90 range: [0.000, 1.619]
  Q1-Q3: [0.000, 0.591]
B6 regions:
  Median: 0.565
  IQR: 1.308
  10-90 range: [0.251, 3.031]
  Q1-Q3: [0.340, 1.648]
Median difference: 0.475

State 3 (Noise):
CAST regions:
  Median: 0.087
  IQR: 0.000
  10-90 range: [0.087, 0.087]
  Q1-Q3: [0.087, 0.087]
B6 regions:
  Median: 0.087
  IQR: 0.000
  10-90 range: [0.087, 0.087]
  Q1-Q3: [0.087, 0.087]

Cell Distribution:
State 1 (CAST > B6): 3141 cells (32.5%)
  Median confidence: 0.881
  IQR confidence: 0.000
State 2 (B6 > CAST): 6218 cells (64.4%)
  Median confidence: 0.881
  IQR confidence: 0.000
State 3 (Noise): 299 cells (3.1%)
  Median confidence: 1.000
  IQR confidence: 0.000


### Save Results

Save the cell state assignments for downstream analysis.

In [ ]:
# Create DataFrame with cell state assignments
cell_states_df = pd.DataFrame({
    'Cell_State': cell_states
}, index=cell_names)

# Add confidence scores (maximum responsibility)
cell_states_df['Confidence'] = responsibilities.max(axis=1)

# Add state labels
state_labels = {0: 'CAST>B6', 1: 'B6>CAST', 2: 'Noise'}
cell_states_df['State_Label'] = cell_states_df['Cell_State'].map(state_labels)

# Display sample
print("\nSample of results:")
print(cell_states_df.head(10))

# Save to file
output_file = OUTPUT_DIR + 'cell_states_bi_ESC_new.csv'
cell_states_df.to_csv(output_file)
print(f"\nResults saved to: {output_file}")

---
## 4. Control Analysis (Annotation Shuffling) {#control-analysis}

### Rationale

To validate that the EM algorithm is detecting real biological signal rather than technical artifacts, we perform a control analysis by shuffling the region type annotations.

### Shuffling Strategy

The shuffling preserves the paired structure of CAST-early and B6-early regions:
1. Divide regions into two halves (representing allelic pairs)
2. Randomly shuffle the labels in the first half
3. Assign complementary labels to the second half (0→1, 1→0)

This maintains the overall balance of region types while disrupting the true biological signal.

In [ ]:
# Shuffle region type annotations
print("Creating shuffled control...")

# Get first and second halves
first_part = region_types[:len(region_types)//2]
second_part = region_types[len(region_types)//2:]

# Shuffle first half randomly
np.random.seed(42)  # Set seed for reproducibility
shuffled_first_part = np.random.permutation(first_part)

# Create complementary second half
shuffled_second_part = 1 - shuffled_first_part

# Combine to get shuffled region types
shuffled_region_types = np.concatenate([shuffled_first_part, shuffled_second_part])

print(f"Original - CAST-early: {(region_types == 1).sum()}, B6-early: {(region_types == 0).sum()}")
print(f"Shuffled - CAST-early: {(shuffled_region_types == 1).sum()}, B6-early: {(shuffled_region_types == 0).sum()}")

Creating shuffled control...
Original - CAST-early: 305, B6-early: 305
Shuffled - CAST-early: 305, B6-early: 305


### Run EM Algorithm on Shuffled Data

Execute the same analysis with shuffled annotations to establish a null distribution.

In [ ]:
# Run analysis on shuffled data
print("\nRunning EM algorithm on shuffled annotations...")
print("=" * 60)
mixing_proportions_shuf, lambda_params_shuf, responsibilities_shuf, cell_states_shuf = analyze_atac_seq_data(
    data=data,
    region_types=shuffled_region_types
)


Running EM algorithm on shuffled annotations...


/tmp/ipython-input-2721987829.py:95: RuntimeWarning: invalid value encountered in subtract
  log_responsibilities -= log_norm



Robust State Analysis:

State 1 (CAST > B6):
CAST regions:
  Median: 0.554
  IQR: 1.204
  10-90 range: [0.254, 2.987]
  Q1-Q3: [0.335, 1.540]
B6 regions:
  Median: 0.079
  IQR: 0.634
  10-90 range: [0.000, 1.551]
  Q1-Q3: [0.000, 0.634]
Median difference: 0.475

State 2 (B6 > CAST):
CAST regions:
  Median: 0.089
  IQR: 0.601
  10-90 range: [0.000, 1.580]
  Q1-Q3: [0.000, 0.601]
B6 regions:
  Median: 0.568
  IQR: 1.281
  10-90 range: [0.270, 3.065]
  Q1-Q3: [0.343, 1.624]
Median difference: 0.479

State 3 (Noise):
CAST regions:
  Median: 0.087
  IQR: 0.000
  10-90 range: [0.087, 0.087]
  Q1-Q3: [0.087, 0.087]
B6 regions:
  Median: 0.087
  IQR: 0.000
  10-90 range: [0.087, 0.087]
  Q1-Q3: [0.087, 0.087]

Cell Distribution:
State 1 (CAST > B6): 3924 cells (40.6%)
  Median confidence: 0.881
  IQR confidence: 0.000
State 2 (B6 > CAST): 5436 cells (56.3%)
  Median confidence: 0.881
  IQR confidence: 0.000
State 3 (Noise): 298 cells (3.1%)
  Median confidence: 1.000
  IQR confidence: 0.000


### Save Control Results

Save both the shuffled region types and resulting cell states.

In [ ]:
# Save shuffled cell states
cell_states_shuf_df = pd.DataFrame({
    'Cell_State': cell_states_shuf,
    'Confidence': responsibilities_shuf.max(axis=1)
})
output_file_shuf = OUTPUT_DIR + 'cell_states_bi_ESC_shuf.csv'
cell_states_shuf_df.to_csv(output_file_shuf, index=False)
print(f"Shuffled cell states saved to: {output_file_shuf}")

# Save shuffled region types for reference
shuffled_region_types_df = pd.DataFrame({
    'Region_Type': shuffled_region_types
})
output_file_regions = OUTPUT_DIR + 'shuffled_region_types.csv'
shuffled_region_types_df.to_csv(output_file_regions, index=False)
print(f"Shuffled region types saved to: {output_file_regions}")

---
## 5. Results and Output {#results}

### Output Files

This notebook generates the following output files:

1. **cell_states_bi_ESC_new.csv**: Cell state assignments from real data
   - Columns: Cell_State (0/1/2), Confidence, State_Label
   - Row names: Cell identifiers

2. **cell_states_bi_ESC_shuf.csv**: Cell state assignments from shuffled control
   - Columns: Cell_State, Confidence
   - No row names (sequential order)

3. **shuffled_region_types.csv**: The shuffled region annotations used in control
   - Single column with shuffled region types
